# Environment check

**Objective.** Verify the installation, versions and test suite.

**Data.** The analysis-ready PISA 2025 public-use extracts are in `data/processed/` (see the Data section of `README.md`), so cloning the repository in Colab is enough to run the real analyses; `DEMO` switches to real data automatically. Set `DEMO = True` for a fast synthetic demonstration. **Demo outputs are not findings.** The *saved results* cells display the PISA 2025 aggregate outputs committed in `results/`.

Errors are raised, never swallowed. Every output is written to `results/` with the form name; outputs from synthetic data are prefixed `demo_`.

In [ ]:
# --- Environment (Colab or local) -------------------------------------------
REPO_URL = ""   # GitHub repository; a private repository needs a token in the URL or an uploaded copy
DEMO = None            # None = real PISA 2025 data if data/processed/ is present (it is in the repository), else synthetic; or force True/False
import os, sys, subprocess, platform
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("ability-adjusted-mcq-profiling"):
        if not REPO_URL:
            raise SystemExit("Set REPO_URL or upload the repository folder to /content.")
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("ability-adjusted-mcq-profiling")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.exists("configs") and os.getcwd() != os.path.dirname(os.getcwd()):
        os.chdir("..")
sys.path.insert(0, "src"); sys.path.insert(0, "scripts")
if DEMO is None:
    DEMO = not os.path.exists("data/processed/math_formA.parquet")
import numpy, scipy, pandas, sklearn
print(platform.python_version(), "numpy", numpy.__version__, "scipy", scipy.__version__,
      "pandas", pandas.__version__, "sklearn", sklearn.__version__)
FORM = "demo" if DEMO else "math_formA"
print("Mode:", "SYNTHETIC DEMONSTRATION - not empirical results" if DEMO else "REAL PISA 2025 data")

In [ ]:
import subprocess, sys
def run(stage, form=None):
    cmd = [sys.executable, "scripts/run_experiments.py", "--stage", stage] + (["--demo"] if DEMO else []) \
          + (["--form", form] if form else [])
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-6000:])
    if r.returncode:
        print(r.stderr[-4000:]); raise RuntimeError(f"stage {stage} failed")

In [ ]:
r = subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests'], capture_output=True, text=True); print(r.stdout[-3000:])